In [14]:
import pandas as pd
from sklearn import neighbors
from sklearn.metrics import recall_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
import numpy as np
import seaborn as sns


In [15]:
training_data1 = pd.read_csv('dados/dados_modelo1.csv') #semanas 1 a 4(ou 5)
training_data2 = pd.read_csv('dados/dados_modelo2.csv') #semanas 1 a 8
training_data3 = pd.read_csv('dados/dados_modelo3.csv') #semanas 1 a 12

# cuidando dos NaN da coluna TempoQ3
for df in [training_data1, training_data2, training_data3]:
    if 'TempoQ3' in df.columns:
        df['TempoQ3'] = df['TempoQ3'].fillna(df['TempoQ3'].mean())


In [16]:
k = 4

def treinar_knn_oversample(df, test_size=0.3, random_state=42, k=k):
    X = df.drop(columns=['Aprovou_Aprovou'])
    y = df['Aprovou_Aprovou']

    # split estratificado
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=random_state
    )

    # normalizar
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    # aplicar SMOTE só no treino
    smote = SMOTE(random_state=random_state)
    X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

    # treinar KNN
    clf = neighbors.KNeighborsClassifier(n_neighbors=k, weights='distance')
    clf.fit(X_train_res, y_train_res)

    return clf, X_train_res, X_test, y_train_res, y_test


In [17]:
clf1, X1_train_res, X1_test, y1_train_res, y1_test = treinar_knn_oversample(training_data1)
clf2, X2_train_res, X2_test, y2_train_res, y2_test = treinar_knn_oversample(training_data2)
clf3, X3_train_res, X3_test, y3_train_res, y3_test = treinar_knn_oversample(training_data3)

# Avaliação
for i, (clf, X_test, y_test) in enumerate([
    (clf1, X1_test, y1_test),
    (clf2, X2_test, y2_test),
    (clf3, X3_test, y3_test)
], start=1):
    y_pred = clf.predict(X_test)
    print(f"Recall Modelo {i}:", recall_score(y_test, y_pred, pos_label=0))
    print(classification_report(y_test, y_pred))


Recall Modelo 1: 0.5416666666666666
              precision    recall  f1-score   support

         0.0       0.22      0.54      0.31        24
         1.0       0.97      0.88      0.92       389

    accuracy                           0.86       413
   macro avg       0.59      0.71      0.62       413
weighted avg       0.93      0.86      0.89       413

Recall Modelo 2: 0.5
              precision    recall  f1-score   support

         0.0       0.24      0.50      0.32        24
         1.0       0.97      0.90      0.93       389

    accuracy                           0.88       413
   macro avg       0.60      0.70      0.63       413
weighted avg       0.92      0.88      0.90       413

Recall Modelo 3: 0.875
              precision    recall  f1-score   support

         0.0       0.36      0.88      0.51        24
         1.0       0.99      0.90      0.94       389

    accuracy                           0.90       413
   macro avg       0.67      0.89      0.73     